# Training

Trains our from-scratch YOLO26 (`src/`) on a chosen dataset through the Ultralytics engine.

## 1. Dependencies

This cell installs and checks if all necessary dependencies are installed.
If not execute in the root directory ``pip install -r requirments.txt``

In [1]:
import importlib.util
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

REQUIRED = ["ultralytics", "yaml"]

if importlib.util.find_spec("torch") is None:
    raise RuntimeError(
        "torch is not installed. Install it from https://pytorch.org/get-started/locally/ "
        "with the index URL matching your CUDA version"
    )

missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]
if missing:
    print("Missing:", " ".join(missing))
else:
    print("All dependencies installed.")

import torch
import ultralytics

print(f"torch {torch.__version__}   ultralytics {ultralytics.__version__}")
print(f"cuda {torch.version.cuda or 'n/a'}   available {torch.cuda.is_available()}")

All dependencies installed.
torch 2.13.0   ultralytics 8.4.120
cuda n/a   available False


## 2. Setup

Import the Ultralytics adapter from the source.

In [2]:
import sys
from pathlib import Path

import yaml


def repo_root(start: Path | None = None) -> Path:
    """Walk up from the notebook until the directory holding the model package appears."""
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "src" / "model.py").exists() or (cand / "model.py").exists():
            return cand
    raise FileNotFoundError(f"no repo root above {here}")


ROOT = repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT.parent) not in sys.path:
    sys.path.insert(0, str(ROOT.parent))

print("repo root:", ROOT)


def import_adapter():
    """Import the Ultralytics adapter."""
    import importlib

    tried = []
    for name in ("src.ultralytics_adapter", "ultralytics_adapter", f"{ROOT.name}.ultralytics_adapter"):
        try:
            return importlib.import_module(name)
        except ImportError as exc:
            tried.append(f"  {name:40s} -> {exc}")
    raise ImportError(
        "could not import ultralytics_adapter. Tried:\n" + "\n".join(tried) +
        "\n\nThe adapter defines MyDetectionModel/MyTrainer/MyYOLO and must sit"
        "\nnext to model.py. This checkout only has its compiled __pycache__ copy."
    )


adapter = import_adapter()
MyYOLO = adapter.MyYOLO
print("adapter:", adapter.__name__)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(0) if DEVICE == 0 else "cpu")

repo root: /Users/valentin/Desktop/yolo26_scratch
adapter: src.ultralytics_adapter
device: cpu


## 3. Select the dataset

Set `DATA` to the `dataset.yaml` of the dataset you want to train on.

In [3]:
# --- CHANGE ME -----------------------------------------------------------------------
DATA = Path(r"C:\Datasets\coco_street_objects_yolo\dataset.yaml")
# -------------------------------------------------------------------------------------

assert DATA.exists(), f"dataset yaml not found: {DATA}"

cfg = yaml.safe_load(DATA.read_text())
cfg["path"] = str(DATA.parent)
DATA.write_text(yaml.safe_dump(cfg, sort_keys=False))

print("dataset:", DATA)
print("classes:", cfg["names"])
for split in ("train", "val"):
    d = DATA.parent / cfg.get(split, "")
    n = sum(1 for _ in d.glob("*")) if d.is_dir() else "?"
    print(f"  {split:5s} {d}  ({n} files)")

AssertionError: dataset yaml not found: C:\Datasets\coco_street_objects_yolo\dataset.yaml

## 4. Training settings

`BATCH` needs to be changed if the GPU runs out of VRAM.

Every epoch is saved to `result/<RUN_NAME>/weights/` (`save_period=1`), next to the usual
`last.pt` and `best.pt`, so `inference.ipynb` can load any point of the schedule.

In [ ]:
RUN_NAME = "my_yolo_training"
EPOCHS = 100
IMGSZ = 640
BATCH = 0.8
FRACTION = 1
WORKERS = 4 if os.name == "nt" else 8

RESULT = ROOT / "result"
RESULT.mkdir(exist_ok=True)

TRAIN_ARGS = dict(
    data=str(DATA),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    fraction=FRACTION,
    optimizer="MuSGD",   # "auto" would pick AdamW on short schedules
    patience=20,
    close_mosaic=10,     # last epochs without mosaic, like the images at inference
    amp=True,
    project=str(RESULT),  # -> result/<name>/
    name=RUN_NAME,
    save_period=1,        # a checkpoint per epoch
    plots=True,
    val=True,
)

print(f"output -> {RESULT / RUN_NAME / 'weights'}")
for k, v in TRAIN_ARGS.items():
    print(f"  {k:12s} {v}")

## 5. Train

Starts the run with the dataset and settings above. From-scratch detection needs a long
schedule, so expect near-zero mAP for the first epochs -- that is not a bug.

In [ ]:
_clip = torch.nn.utils.clip_grad_norm_
if not getattr(_clip, "_low_mem", False):
    def clip_grad_norm_low_mem(parameters, max_norm, norm_type=2.0,
                               error_if_nonfinite=False, foreach=None):
        return _clip(parameters, max_norm, norm_type, error_if_nonfinite, foreach=False)

    clip_grad_norm_low_mem._low_mem = True
    torch.nn.utils.clip_grad_norm_ = clip_grad_norm_low_mem

model = MyYOLO()
results = model.train(**TRAIN_ARGS)

RUN_DIR = Path(results.save_dir)
print("\nrun dir:", RUN_DIR)
print("weights:", sorted(p.name for p in (RUN_DIR / "weights").glob("*.pt")))

## 6. Validate

`map50-95` is the strict COCO metric, `map50` the looser IoU=0.5 one.

In [ ]:
metrics = model.val(data=str(DATA), imgsz=IMGSZ, batch=max(1, BATCH // 2), device=DEVICE)

print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}\n")
for i, name in cfg["names"].items():
    print(f"  {name:12s} AP50-95: {metrics.box.maps[i]:.4f}")

## Resuming an interrupted run

A checkpoint changes the model object, so `last.pt` carries the architecture with it --
but stock `YOLO.train()` would hand its minimal `yaml` to `parse_model` and raise
`KeyError: 'backbone'`. Use `MyYOLO` to continue:

```python
model = MyYOLO(RUN_DIR / "weights" / "last.pt")
model.train(resume=True)
```